In [55]:
import multiprocessing
import numpy as np
from functions import *
import itertools
from pprint import pprint

In [3]:
multiprocessing.cpu_count()

4

In [43]:
reso = 3
z = 1

logMstar0 = np.linspace(8.0,12.0,reso)
xsigpre = np.linspace(1.0,4.0,reso)
xsigpost = np.linspace(1.0,4.0,reso)
slopes = np.linspace(0.01,1.3,reso)
norms = np.linspace(0.5,3.0,reso)
local_norms = np.linspace(np.log10(0.001), np.log10(0.005), reso)

Lleft = 8.95
Lright = 14.95
points_to_fit = 100
midpoint = points_to_fit//2+points_to_fit%2

lums = np.zeros(points_to_fit)
    
_, _, _, _, Lbreak = Shen_fit_uncer(z, lums)
    
lums[0:midpoint] = np.linspace(Lleft, Lbreak, midpoint, endpoint = False)
lums[midpoint:] = np.linspace(Lbreak, Lright, points_to_fit//2)

print('\nCreating combination array.')

combos = np.array(list(itertools.product(logMstar0, xsigpre, xsigpost, slopes, norms, local_norms)))

print('Done.')


Creating combination array.
Done.


In [44]:
def chi2(a, qlf):
    qlf.get_Mbh(a[0], a[3], a[4], approx_local=True, norm_local = 11+a[5])
    qlf.get_dNdlnL(lums, [a[1], a[2]])

    ym = np.log10(qlf.dNdlogL)
    presum = (ym-ya)**2
    return np.sum((ym-ya)**2)

def process_chunk(all_args):
    
    func, axis, combo_slice, z = all_args
    
    qlf_bins = 0.005
    qlf = QLF(z, qlf_bins)
    
    return np.apply_along_axis(chi2, axis, combo_slice, qlf)

In [45]:
ya, err_ave, err_abv, err_blw, _ = Shen_fit_uncer(z, lums)

chunks = [(chi2, 1, combo_slice, z)
              for combo_slice in np.array_split(combos, multiprocessing.cpu_count())]

pool = multiprocessing.Pool()
chunk_results = pool.map(process_chunk, chunks)

# Freeing the workers:
pool.close()
pool.join()


final = np.concatenate(chunk_results).reshape((reso, reso, reso, reso, reso, reso))

In [46]:
qlf_bins = 0.005
z = 1.0
qlf = QLF(z, qlf_bins)
    
normal_result = np.apply_along_axis(chi2, 1, combos, qlf).reshape((reso, reso, reso, reso, reso, reso))

In [60]:
r1, r2, r3, r4, r5, r6 = 2, 2, 1, 2, 2, 2
z = 1
qlf_bins = 0.005

logMstar0 = np.linspace(1,2,r1)
xsigpre = np.linspace(3,4,r2)
xsigpost = np.linspace(5,5,r3)
slopes = np.linspace(6,7,r4)
norms = np.linspace(8,9,r5)
local_norms = np.linspace(10,11, r6)

Lleft = 8.95
Lright = 14.95
points_to_fit = 100
midpoint = points_to_fit//2+points_to_fit%2

lums = np.zeros(points_to_fit)
    
_, _, _, _, Lbreak = Shen_fit_uncer(z, lums)
    
lums[0:midpoint] = np.linspace(Lleft, Lbreak, midpoint, endpoint = False)
lums[midpoint:] = np.linspace(Lbreak, Lright, points_to_fit//2)

print('\nCreating combination array.')

combos = np.array(list(itertools.product(logMstar0, xsigpre, xsigpost, slopes, norms, local_norms)))

print('Done.')
pprint(combos.reshape((r1,r2,r3,r4,r5,r6,6)))

ya, err_ave, err_abv, err_blw, _ = Shen_fit_uncer(z, lums)

chunks = [(chi2, 1, combo_slice, z)
              for combo_slice in np.array_split(combos, multiprocessing.cpu_count())]

pool = multiprocessing.Pool()
chunk_results = pool.map(process_chunk, chunks)

# Freeing the workers:
pool.close()
pool.join()


final = np.concatenate(chunk_results).reshape((r1,r2,r3,r4,r5,r6))
print(final)

qlf = QLF(z, qlf_bins)
    
normal_result = np.apply_along_axis(chi2, 1, combos, qlf).reshape((r1,r2,r3,r4,r5,r6))
print(final == normal_result)


Creating combination array.
Done.
array([[[[[[[ 1.,  3.,  5.,  6.,  8., 10.],
            [ 1.,  3.,  5.,  6.,  8., 11.]],

           [[ 1.,  3.,  5.,  6.,  9., 10.],
            [ 1.,  3.,  5.,  6.,  9., 11.]]],


          [[[ 1.,  3.,  5.,  7.,  8., 10.],
            [ 1.,  3.,  5.,  7.,  8., 11.]],

           [[ 1.,  3.,  5.,  7.,  9., 10.],
            [ 1.,  3.,  5.,  7.,  9., 11.]]]]],




        [[[[[ 1.,  4.,  5.,  6.,  8., 10.],
            [ 1.,  4.,  5.,  6.,  8., 11.]],

           [[ 1.,  4.,  5.,  6.,  9., 10.],
            [ 1.,  4.,  5.,  6.,  9., 11.]]],


          [[[ 1.,  4.,  5.,  7.,  8., 10.],
            [ 1.,  4.,  5.,  7.,  8., 11.]],

           [[ 1.,  4.,  5.,  7.,  9., 10.],
            [ 1.,  4.,  5.,  7.,  9., 11.]]]]]],





       [[[[[[ 2.,  3.,  5.,  6.,  8., 10.],
            [ 2.,  3.,  5.,  6.,  8., 11.]],

           [[ 2.,  3.,  5.,  6.,  9., 10.],
            [ 2.,  3.,  5.,  6.,  9., 11.]]],


          [[[ 2.,  3.,  5.,  7.,  8., 10.],
 